# Phase 3 RQ1 v2 — combined analysis

This analysis stage runs on CPU; no model training or GPU is needed. It analyzes the same ten-language Phase 4 roster: `en`, `de`, `es`, `fr`, `sw`, `ar`, `ur`, `bg`, `ru`, and `hi`. Turn Internet on to clone the repository. Attach the private input dataset containing `phase3_rq1_v2_inputs.zip` and the 15 per-order result ZIPs. You may attach them individually, or attach one outer ZIP containing the input ZIP and all 15 result ZIPs; the notebook extracts that format automatically. The workflow also accepts one complete seed ZIP instead of five per-order ZIPs, but every seed must have all five orders represented exactly once.

The workflow checks that the seeds, five orders per seed, 15 result records, and data-manifest hashes match the frozen protocol before calculating pooled and per-order correlations, 10,000 language-label permutations, Holm correction, and a random-score control. It exports `phase3_rq1_v2_analysis.zip` under `/kaggle/working`. Review the per-seed metric summaries in the provenance and the per-order correlation table alongside the pooled corrected results; this pilot does not automatically select a prior.

In [ ]:
from pathlib import Path
import shutil
import subprocess
import sys
from zipfile import ZipFile

REPO_URL = "https://github.com/Mani-S-2709/asymmetric-interference-continual-learning..git"
REPO_ROOT = Path("/kaggle/working/asymmetric-interference-continual-learning")
if not REPO_ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_ROOT)], check=True)
elif (REPO_ROOT / ".git").is_dir():
    subprocess.run(["git", "-C", str(REPO_ROOT), "remote", "set-url", "origin", REPO_URL], check=True)
    subprocess.run(["git", "-C", str(REPO_ROOT), "pull", "--ff-only"], check=True)
else:
    raise FileExistsError(f"Repository path exists but is not a Git clone: {REPO_ROOT}")

sys.path.insert(0, str(REPO_ROOT))
INPUT_ROOT = Path("/kaggle/input")
INPUT_ZIP_NAME = "phase3_rq1_v2_inputs.zip"
RESULT_PREFIX = "phase3_rq1_v2_b0_seed_"
inputs = list(INPUT_ROOT.rglob(INPUT_ZIP_NAME))
if not inputs:
    input_summaries = list(INPUT_ROOT.rglob("rq1_v2_input_summary.json"))
    if len(input_summaries) == 1:
        input_root = input_summaries[0].parent
        input_bundle = Path("/kaggle/working") / INPUT_ZIP_NAME
        with ZipFile(input_bundle, "w") as archive:
            for path in sorted(input_root.rglob("*")):
                if path.is_file():
                    archive.write(path, path.relative_to(input_root).as_posix())
        inputs = [input_bundle]
        print(f"Repacked the extracted input dataset: {input_bundle}")
    elif len(input_summaries) > 1:
        raise ValueError(f"Expected one extracted input dataset; found: {input_summaries}")

if not inputs:
    embedded_inputs = []
    for outer_zip in INPUT_ROOT.rglob("*.zip"):
        with ZipFile(outer_zip) as archive:
            members = [name for name in archive.namelist() if Path(name).name == INPUT_ZIP_NAME]
            for member in members:
                extracted_input = Path("/kaggle/working") / INPUT_ZIP_NAME
                with archive.open(member) as source, extracted_input.open("wb") as target:
                    shutil.copyfileobj(source, target)
                embedded_inputs.append(extracted_input)
    inputs = embedded_inputs

if len(inputs) != 1:
    visible_files = [str(path) for path in INPUT_ROOT.rglob("*") if path.is_file()]
    raise FileNotFoundError(
        "Attach exactly one dataset containing phase3_rq1_v2_inputs.zip, "
        "its extracted files, or an outer ZIP that contains it. "
        f"Found input bundles: {inputs}. Files under /kaggle/input: {visible_files}"
    )

result_zips = sorted(INPUT_ROOT.rglob(f"{RESULT_PREFIX}*.zip"))
result_names = {path.name for path in result_zips}
extracted_results = Path("/kaggle/working/phase3_rq1_v2_result_zips")
extracted_results.mkdir(parents=True, exist_ok=True)
for outer_zip in INPUT_ROOT.rglob("*.zip"):
    with ZipFile(outer_zip) as archive:
        for member in archive.namelist():
            name = Path(member).name
            if name.startswith(RESULT_PREFIX) and name.endswith(".zip") and name not in result_names:
                result_path = extracted_results / name
                with archive.open(member) as source, result_path.open("wb") as target:
                    shutil.copyfileobj(source, target)
                result_zips.append(result_path)
                result_names.add(name)
result_zips = sorted(result_zips)

if len(result_zips) < 3 or len(result_zips) > 15:
    visible_files = [str(path) for path in INPUT_ROOT.rglob("*") if path.is_file()]
    raise FileNotFoundError(
        "Attach either three complete seed ZIPs or 15 per-order result ZIPs, "
        f"individually or inside one outer ZIP. Found {len(result_zips)}: {result_zips}. "
        f"Files under /kaggle/input: {visible_files}"
    )

OUTPUT_DIR = Path("/kaggle/working/phase3_rq1_v2_analysis")
command = [
    sys.executable, "-m", "phase3.rq1_v2_workflow", "analyze",
    "--inputs-zip", str(inputs[0]),
    "--output-dir", str(OUTPUT_DIR),
    "--random-repetitions", "10000",
]
for result_zip in result_zips:
    command.extend(["--result-zip", str(result_zip)])
subprocess.run(command, cwd=REPO_ROOT, check=True)

ANALYSIS_ZIP = OUTPUT_DIR / "phase3_rq1_v2_analysis.zip"
if not ANALYSIS_ZIP.is_file():
    raise FileNotFoundError(f"Analysis bundle was not created: {ANALYSIS_ZIP}")
print(f"Download the combined analysis from Kaggle output files: {ANALYSIS_ZIP}")